# Bagging for Decision Trees


We observed with the Titanic data example that decision trees are unstable. That is, theychange substantially between different training subsets of the same data and even minor differences in algorithm. 

This instability is not a bug but a structural feature of the greedy, recursive splitting procedure. Each split is chosen to maximize local purity, and small perturbations in the data can shift which variable and threshold is selected — propagating instability all the way down the tree.

### Core Idea of Bagging
If we had many independent sets of data from the same data generating process, we could fit a tree to each one and then average the results in some way. This would hopefully average out the random variation and give a more reliable tree. The core idea of bagging is to use bootstrap samples as a way to provide multiple data sets. They won't be independent, but we hope that averaging over them will still produce more reliable trees. 


### Bagging Algorithm

Input: Training data $D$, base learner (decision tree), number of trees B, tree depth or stopping rule.

**For b = 1, 2, ..., B:  
  {
      Draw a bootstrap sample $D_b$ from $D$ (size n, with replacement).   
              Fit a deep (low-bias) decision tree $T_b$ on $D_b$, applying minimal or no pruning.   
   }**
   
   
**Prediction (classification): Take the majority vote across trees:**

The predicted class for input $x$ is 

$$\hat{g(x)}=\arg \max_k \sum_{b=1}^B I(T_b(x)=k)$$


**Prediction (regression): Average the predictions:**

$$f_{bag}(x)=\frac{1}{B} \sum_{b=1}^{B} T_b(x)$$



### Application to Titanic Data

Next, we will demonstrate the use of bagging with the Titanic data. Import modules, read in the Titanic data, and set up test and training data. This is the version of the data that we used for making trees in the previous notes. 

In [20]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd


titanic=pd.read_csv("TitanicFinal.csv")
titanic



,survived,pclass,sex,age,fare,embarked,FamilySize
0,1,1,0,29.0000,3.0,0,0
1,1,1,1,0.9167,3.0,0,3
2,0,1,0,2.0000,3.0,0,3
3,0,1,1,30.0000,3.0,0,3
4,0,1,0,25.0000,3.0,0,3
...,...,...,...,...,...,...,...
1304,0,3,0,14.5000,2.0,1,1
1305,0,3,0,NaN,2.0,1,1
1306,0,3,1,26.5000,0.0,1,0
1307,0,3,1,27.0000,0.0,1,0


Set up the training and test data:

In [21]:
X = titanic.drop(columns=["survived"])
y = titanic["survived"]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=12)

Fit a single tree: 

In [2]:
single_tree = DecisionTreeClassifier(random_state=12)
single_tree.fit(X_train, y_train)
tree_preds = single_tree.predict(X_test)
tree_acc = accuracy_score(y_test, tree_preds)
tree_acc

0.732824427480916

In [3]:
print(classification_report(y_test,tree_preds))

              precision    recall  f1-score   support

           0       0.77      0.81      0.79       159
           1       0.67      0.62      0.65       103

    accuracy                           0.73       262
   macro avg       0.72      0.71      0.72       262
weighted avg       0.73      0.73      0.73       262



We use the `sklearn` function `BaggingClassifier` to do bagging: 

In [7]:
bagged_trees = BaggingClassifier(
    estimator=DecisionTreeClassifier(),  # base learner
    n_estimators=100,                    # number of trees
    max_samples=1.0,                     # bootstrap sample size (fraction of training set)
    max_features=1.0,                    # fraction of features per tree
    bootstrap=True,                      # sample with replacement
    bootstrap_features=False,            # don't resample features (that's Random Forest territory)
    oob_score=False,                      # out-of-bag error estimate
    random_state=42,
    n_jobs=-1
)

bagged_trees.fit(X_train, y_train)
bag_preds = bagged_trees.predict(X_test)
bag_acc = accuracy_score(y_test, bag_preds)
bag_acc

0.7633587786259542

In [8]:
print(classification_report(y_test,bag_preds))

              precision    recall  f1-score   support

           0       0.80      0.82      0.81       159
           1       0.71      0.68      0.69       103

    accuracy                           0.76       262
   macro avg       0.75      0.75      0.75       262
weighted avg       0.76      0.76      0.76       262



In this example, the bagged tree has slightly higher accuracy than a single tree. 

### Out of Bag Estimates

It can be shown tha a bootstrap sample will contain 63% of the data on average. This means that an average of 37% of samples will be included in each bootstrap sample. This gives a free way to estimate prediction error. For any observation $(x_i,y_i)$, there will be some subset of trees - those not trained on that observation — for which $(x_i,y_i)$ is genuinely out-of-sample. We can use those trees to predict $y_i$ and assess prediction quality.

**out of bag prediction**
    
For each training observation $i$:
1. Collect only the trees for which observation $i$ was not in the bootstrap sample
2. Have those trees vote on the class of $i$
3. The majority vote is the OOB prediction for $i$

Once every training observation has an OOB prediction, OOB error is just the misclassification rate across all of them:

**OOB Error=proportion of training observations misclassified by the OOB trees**

Why Out of Bag Error is useful:
1. It's essentially a free cross-validation estimate — we don't have to hold any data out or do substantial extra computation. 
2. As $B$ gets large, the OOB coverges to leave-one-out cross validation. 
3. It's useful for tuning the paramter `n_estimators` — you can plot OOB error vs. number of trees and stop when it flattens out:

We can get the OOB score by setting the `oob_score` option to `True`. The score is then contained in the attribute `oob_score_`. 

In [11]:
bagged_trees = BaggingClassifier(
    estimator=DecisionTreeClassifier(),  # base learner
    n_estimators=100,                    # number of trees
    max_samples=1.0,                     # bootstrap sample size (fraction of training set)
    max_features=1.0,                    # fraction of features per tree
    bootstrap=True,                      # sample with replacement
    bootstrap_features=False,            # don't resample features (that's Random Forest territory)
    oob_score=True,                      # out-of-bag error estimate
    random_state=42,
    n_jobs=-1
)

bagged_trees.fit(X_train, y_train)
bagged_trees.oob_score_

0.7879656160458453

### Random Forests

The reason why bagging is beneficial is clear: averaging over many bootstrapped trees will tend to average out random variation and get better predictions. A fundamental challenge, however, is that the bagged trees are not independent. In virtually any real dataset, the predictors are not equally informative. There will typically be one or a few features that explain a disproportionate share of the variance in the response.  When a deep tree is grown greedily, the algorithm evaluates all features at every internal node and selects the split that maximizes impurity reduction. A dominant variable will win this competition at the root — and very likely at many subsequent nodes — across essentially every bootstrap sample.

The result is that all $B$ trees share the same high-level structure: the dominant feature appears at the top of every tree, partitioning the feature space in roughly the same way before any deeper variation can emerge. The bootstrap resampling changes which observations are used, but it cannot change the fact that the dominant variable is dominant. The trees are correlated because they are all, in essence, expressions of the same strong signal.


This means that errors in the trees will tend not to be averaged out by bagging. 

**The core idea of Random Forests: at each node in each tree, rather than searching over all $p$ features for the best split, the algorithm randomly samples $m < p$ features and searches only within that restricted set.**

**Random Forests Algorithm**
  
**For b = 1, 2, ..., B:  
  {
    1. Draw a bootstrap sample $D_b$ from $D$ (size n, with replacement).   
    2. Grow a tree $T_b$ on $D_b$ by the following recursive procedure until nodes reach size $n_{min}$:  
        -At the current node, draw a random subset of $m$ features from the full $p$ features.  
        -Find the best split using only those $m$ features.  
        -Split the node and recurse on the two children.   
   }**
   
   
**Prediction (classification): Take the majority vote across trees:**


#### Example: Random Forest for the Titanic Data

We use the `sklearn` function `RandomForestClassifier` to do random forests. The procedures and sytax are similar to above.      

In [18]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,      # number of trees
    max_depth=None,        # grow trees fully unless limited
    max_features="sqrt",   # features considered at each split (sqrt is the classic RF setting)
    min_samples_split=2,   
    random_state=12,
    n_jobs=-1              # use all CPU cores
)
rf.fit(X_train, y_train)



y_pred = rf.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred))
      


Accuracy: 0.7710

              precision    recall  f1-score   support

           0       0.80      0.83      0.81       159
           1       0.72      0.68      0.70       103

    accuracy                           0.77       262
   macro avg       0.76      0.75      0.76       262
weighted avg       0.77      0.77      0.77       262



**Feature importance** is a measure of how much a feature affects the structure of the tree. It is calculated using out-of-bag estimates.  It is in the same order as the features in the data. 

In [17]:
rf.feature_importances_

array([0.10736019, 0.30241979, 0.37505896, 0.07778963, 0.04268002,
       0.09469141])

In [22]:
X

,pclass,sex,age,fare,embarked,FamilySize
0,1,0,29.0000,3.0,0,0
1,1,1,0.9167,3.0,0,3
2,1,0,2.0000,3.0,0,3
3,1,1,30.0000,3.0,0,3
4,1,0,25.0000,3.0,0,3
...,...,...,...,...,...,...
1304,3,0,14.5000,2.0,1,1
1305,3,0,NaN,2.0,1,1
1306,3,1,26.5000,0.0,1,0
1307,3,1,27.0000,0.0,1,0
